# Shape-Aware StarDist Training on Colab

This notebook sets up the environment and trains the Shape-Aware StarDist model.

## Setup Steps:
1. Mount Google Drive
2. Clone repository
3. Install dependencies
4. Prepare data
5. Train model
6. Save results


In [ ]:
# Check GPU availability
!nvidia-smi


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Clone repository from GitHub
!git clone https://github.com/EttaZhouuuuu/hdrg-adaptive-stardist.git
%cd hdrg-adaptive-stardist
!git checkout feature/shape-aware-backbone


In [ ]:
# Install dependencies
%pip install -q albumentations scikit-image tqdm tensorboard
print("✓ Dependencies installed")


In [ ]:
# Copy data from Drive to Colab local storage (faster training)
!mkdir -p data
!unzip -q /content/drive/MyDrive/shape_data/dsb2018.zip -d data/
print("✓ Data copied to local storage")


In [ ]:
# Import libraries and setup
import torch
import sys
sys.path.insert(0, '/content/hdrg-adaptive-stardist')

from shape_aware_stardist.models.adaptive_shape_encoder import AdaptiveShapeEncoder
from shape_aware_stardist.training.shape_aware_loss import ShapeAwareHybridLoss
from shape_aware_stardist.training.trainer import ShapeAwareTrainer
from shape_aware_stardist.data.dsb2018_dataset import get_dsb2018_loaders
from shape_aware_stardist.tracking.experiment_tracker import ExperimentTracker

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')


In [ ]:
# Load data
print("Loading DSB2018 dataset...")
train_loader, val_loader, test_loader = get_dsb2018_loaders(
    batch_size=8,  # Adjust based on GPU memory
    input_size=(256, 256),
    n_rays=32,
    num_workers=2
)

print(f"Train: {len(train_loader.dataset)} images")
print(f"Val: {len(val_loader.dataset)} images")
print(f"Test: {len(test_loader.dataset)} images")


In [ ]:
# Create model
print("Creating model...")
model = AdaptiveShapeEncoder(
    in_channels=3,
    n_rays=32,
    base_channels=64,
    n_blocks=4,
    use_shape_prior=True
).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


In [ ]:
# Create loss function and optimizer
loss_fn = ShapeAwareHybridLoss(
    n_rays=32,
    consistency_weight=1.0,
    smoothness_weight=0.5,
    adversarial_weight=0.1,
    use_focal_loss=True,
    use_adaptive_weights=True
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=10,
    verbose=True
)

print("✓ Loss function and optimizer created")


In [ ]:
# Create trainer
config = {
    'num_epochs': 200,
    'early_stopping_patience': 30,
    'save_best_only': True,
    'validation_interval': 1
}

trainer = ShapeAwareTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    device=device,
    config=config,
    scheduler=scheduler,
    experiment_dir='/content/experiments'
)

print("✓ Trainer created")


In [ ]:
# Start training
print("="*50)
print("Starting training...")
print("="*50)

history = trainer.train(num_epochs=200)

print("\n" + "="*50)
print("Training completed!")
print("="*50)
print(f"Best validation loss: {history['best_val_loss']:.4f}")


In [ ]:
# Save results to Google Drive
print("Saving results to Google Drive...")
!mkdir -p /content/drive/MyDrive/shape_aware_stardist/trained_models
!cp -r /content/experiments/* /content/drive/MyDrive/shape_aware_stardist/trained_models/
print("✓ Results saved to Drive")
